In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/PetImages/data")

CAT_DIR = DATA_DIR / "Cat"
DOG_DIR = DATA_DIR / "Dog"

print("Data folder exists:", DATA_DIR.exists())
print("Cat folder exists:", CAT_DIR.exists())
print("Dog folder exists:", DOG_DIR.exists())

Data folder exists: True
Cat folder exists: True
Dog folder exists: True


In [ ]:
cat_images = list(CAT_DIR.glob("*.jpg"))
dog_images = list(DOG_DIR.glob("*.jpg"))

print("Cat images:", len(cat_images))
print("Dog images:", len(dog_images))
print("Total images:", len(cat_images) + len(dog_images))

Cat images: 12499
Dog images: 12499
Total images: 24998


In [ ]:
image_paths = []
labels = []

# Cat → 0
for image_path in cat_images:
    image_paths.append(str(image_path))
    labels.append(0)

# Dog → 1
for image_path in dog_images:
    image_paths.append(str(image_path))
    labels.append(1)

print("Total images:", len(image_paths))
print("Total labels:", len(labels))
print("First 5 labels:", labels[:5])

Total images: 24998
Total labels: 24998
First 5 labels: [0, 0, 0, 0, 0]


In [ ]:
from sklearn.model_selection import train_test_split

train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Training images:", len(train_paths))
print("Validation images:", len(val_paths))
print("Training labels:", len(train_labels))
print("Validation labels:", len(val_labels))


Training images: 19998
Validation images: 5000
Training labels: 19998
Validation labels: 5000


In [ ]:
!pip install -q transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
from transformers import AutoImageProcessor, ResNetForImageClassification

model_name = "microsoft/resnet-50"

processor = AutoImageProcessor.from_pretrained(model_name)

model = ResNetForImageClassification.from_pretrained(
    model_name,
    num_labels=2,
    ignore_mismatched_sizes=True
)

# Our two classes
model.config.id2label = {
    0: "CAT",
    1: "DOG"
}

model.config.label2id = {
    "CAT": 0,
    "DOG": 1
}

print("Model loaded successfully!")

preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.6k [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[transformers] ResNetForImageClassification LOAD REPORT from: microsoft/resnet-50
Key                 | Status   |                                                                                            
--------------------+----------+--------------------------------------------------------------------------------------------
classifier.1.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 2048]) vs model:torch.Size([2, 2048])
classifier.1.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])            

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Model loaded successfully!


In [ ]:
device = torch.device("cuda")

model = model.to(device)

print("Model device:", next(model.parameters()).device)
print("GPU:", torch.cuda.get_device_name(0))

Model device: cuda:0
GPU: Tesla T4


In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class CatDogDataset(Dataset):

    def __init__(self, image_paths, labels, processor):
        self.image_paths = image_paths
        self.labels = labels
        self.processor = processor

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        # Load only one image when it is needed
        image = Image.open(self.image_paths[idx]).convert("RGB")

        # Prepare the image for ResNet-50
        processed = self.processor(
            images=image,
            return_tensors="pt"
        )

        # Remove the extra batch dimension
        pixel_values = processed["pixel_values"].squeeze(0)

        # Convert label to a PyTorch tensor
        label = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return {
            "pixel_values": pixel_values,
            "labels": label
        }

print("Dataset class created successfully!")

Dataset class created successfully!


In [ ]:
train_dataset = CatDogDataset(
    train_paths,
    train_labels,
    processor
)

val_dataset = CatDogDataset(
    val_paths,
    val_labels,
    processor
)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

Training dataset: 19998
Validation dataset: 5000


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training batches: 1250
Validation batches: 313


In [ ]:
batch = next(iter(train_loader))

images = batch["pixel_values"].to(device)
labels = batch["labels"].to(device)

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Images device:", images.device)
print("Labels device:", labels.device)

Images shape: torch.Size([16, 3, 224, 224])
Labels shape: torch.Size([16])
Images device: cuda:0
Labels device: cuda:0


In [ ]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=5e-5
)

print("Optimizer created successfully!")

Optimizer created successfully!


In [ ]:
model.train()

total_loss = 0

for batch_index, batch in enumerate(train_loader):

    # Move images and labels to the GPU
    images = batch["pixel_values"].to(device)
    labels = batch["labels"].to(device)

    # Clear previous gradients
    optimizer.zero_grad()

    # Make predictions and calculate loss
    outputs = model(
        pixel_values=images,
        labels=labels
    )

    loss = outputs.loss

    # Calculate gradients
    loss.backward()

    # Update model weights
    optimizer.step()

    total_loss += loss.item()

    # Show progress every 100 batches
    if (batch_index + 1) % 100 == 0:
        print(
            f"Batch {batch_index + 1}/{len(train_loader)} "
            f"| Loss: {loss.item():.4f}"
        )

average_loss = total_loss / len(train_loader)

print("\nTraining complete!")
print("Average training loss:", average_loss)

In [ ]:
import os

model_save_path = "/content/drive/MyDrive/PetImages/cat_dog_resnet50"

os.makedirs(model_save_path, exist_ok=True)

model.save_pretrained(model_save_path)
processor.save_pretrained(model_save_path)

print("Model saved successfully!")
print("Saved to:", model_save_path)

In [ ]:
import torch

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch_index, batch in enumerate(val_loader):

        images = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(pixel_values=images)

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        if (batch_index + 1) % 50 == 0:
            print(
                f"Validation batch {batch_index + 1}/{len(val_loader)}"
            )

accuracy = correct / total

print("\nValidation complete!")
print("Correct predictions:", correct)
print("Total validation images:", total)
print(f"Validation accuracy: {accuracy * 100:.2f}%")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in val_loader:

        images = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(pixel_values=images)

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Classification Report:\n")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["CAT", "DOG"],
        digits=4
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        all_labels,
        all_predictions
    )
)


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import torch

# Pick one validation image
test_path = val_paths[0]
true_label = val_labels[0]

# Load image
image = Image.open(test_path).convert("RGB")

# Prepare image
inputs = processor(
    images=image,
    return_tensors="pt"
)

# Move to GPU
inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

# Predict
model.eval()

with torch.no_grad():
    outputs = model(**inputs)

prediction = torch.argmax(
    outputs.logits,
    dim=1
).item()

# Convert numbers back to names
label_names = {
    0: "CAT",
    1: "DOG"
}

predicted_label = label_names[prediction]
actual_label = label_names[true_label]

# Display
plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.axis("off")
plt.title(
    f"Predicted: {predicted_label} | Actual: {actual_label}"
)
plt.show()

print("Image:", test_path)
print("Predicted:", predicted_label)
print("Actual:", actual_label)

In [ ]:
# Get probabilities for CAT and DOG
probabilities = torch.softmax(
    outputs.logits,
    dim=1
)

cat_probability = probabilities[0][0].item()
dog_probability = probabilities[0][1].item()

predicted_probability = probabilities[0][prediction].item()

print(f"CAT probability: {cat_probability * 100:.2f}%")
print(f"DOG probability: {dog_probability * 100:.2f}%")
print(f"Prediction: {predicted_label}")
print(f"Confidence: {predicted_probability * 100:.2f}%")

In [ ]:
def predict_image(image):

    # Prepare the image
    inputs = processor(
        images=image,
        return_tensors="pt"
    )

    # Move inputs to the GPU
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Make prediction
    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    # Convert logits into probabilities
    probabilities = torch.softmax(
        outputs.logits,
        dim=1
    )

    # Find the class with the highest probability
    predicted_class = torch.argmax(
        probabilities,
        dim=1
    ).item()

    # Get confidence
    confidence = probabilities[0][predicted_class].item()

    # Get class name
    label = model.config.id2label[predicted_class]

    return label, confidence


print("Prediction function created successfully!")

In [ ]:
label, confidence = predict_image(image)

print("Prediction:", label)
print(f"Confidence: {confidence * 100:.2f}%")

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

def classify_image(image):
    label, confidence = predict_image(image)

    return f"{label} — {confidence * 100:.2f}% confidence"


app = gr.Interface(
    fn=classify_image,
    inputs=gr.Image(type="pil", label="Upload a Cat or Dog"),
    outputs=gr.Textbox(label="Prediction"),
    title="🐱 Cat vs Dog Classifier 🐶",
    description="Upload an image and the AI will predict whether it is a cat or a dog."
)

print("App created successfully!")

In [ ]:
app.launch(share=True)

In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/PetImages/data")

print("DATASET STRUCTURE")
print("=" * 60)

for path in sorted(DATA_DIR.rglob("*")):
    if path.is_dir():
        relative = path.relative_to(DATA_DIR)
        print(f"\n📁 {relative}/")

        # Count image files in this folder
        images = [
            f for f in path.iterdir()
            if f.is_file() and f.suffix.lower() in {
                ".jpg", ".jpeg", ".png", ".webp"
            }
        ]

        if images:
            print(f"   Images directly inside: {len(images)}")

print("\n" + "=" * 60)
print("IMAGE COUNTS BY MAIN CLASS")
print("=" * 60)

for main_folder in ["Cat", "Dog", "Neither"]:
    folder = DATA_DIR / main_folder

    if not folder.exists():
        print(f"❌ {main_folder}: NOT FOUND")
        continue

    if main_folder == "Neither":
        # Count images inside all Neither subfolders
        total = 0

        for subfolder in sorted(folder.iterdir()):
            if subfolder.is_dir():
                count = sum(
                    1 for f in subfolder.rglob("*")
                    if f.is_file() and f.suffix.lower() in {
                        ".jpg", ".jpeg", ".png", ".webp"
                    }
                )
                print(f"{subfolder.name:20} {count:>6} images")
                total += count

        print(f"{'NEITHER TOTAL':20} {total:>6} images")

    else:
        count = sum(
            1 for f in folder.rglob("*")
            if f.is_file() and f.suffix.lower() in {
                ".jpg", ".jpeg", ".png", ".webp"
            }
        )

        print(f"{main_folder:20} {count:>6} images")

In [2]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/PetImages/data")
MODEL_DIR = Path("/content/drive/MyDrive/PetImages/cat_dog_resnet50")

print("DATASET EXISTS:", DATA_DIR.exists())
print("MODEL EXISTS:", MODEL_DIR.exists())

print("\nDataset folders:")
for folder in DATA_DIR.iterdir():
    if folder.is_dir():
        print("📁", folder.name)

print("\nSaved model files:")
if MODEL_DIR.exists():
    for file in MODEL_DIR.iterdir():
        print("📄", file.name)

DATASET EXISTS: True
MODEL EXISTS: True

Dataset folders:
📁 Cat
📁 Dog
📁 Neither

Saved model files:
📄 config.json
📄 model.safetensors
📄 preprocessor_config.json


In [3]:
from pathlib import Path

CARS_DIR = Path(
    "/content/drive/MyDrive/PetImages/data/Neither/Cars"
)

CARS_DIR.mkdir(parents=True, exist_ok=True)

car_images = [
    file for file in CARS_DIR.iterdir()
    if file.is_file()
]

print("Cars folder:", CARS_DIR)
print("Existing car images:", len(car_images))

Cars folder: /content/drive/MyDrive/PetImages/data/Neither/Cars
Existing car images: 0


In [4]:
from datasets import load_dataset

cars_dataset = load_dataset(
    "AIOmarRehan/Vehicles",
    split="train",
    streaming=True
)

print("Dataset loaded successfully!")
print("\nFeatures:")
print(cars_dataset.features)

README.md:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

Dataset loaded successfully!

Features:
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['Auto Rickshaws', 'Bikes', 'Cars', 'Motorcycles', 'Planes', 'Ships', 'Trains'])}


In [5]:
CAR_LABEL = "Cars"

label_names = cars_dataset.features["label"].names
car_label_id = label_names.index(CAR_LABEL)

print("Car label ID:", car_label_id)
print("Car label name:", label_names[car_label_id])

# Find the first car image
for example in cars_dataset:
    if example["label"] == car_label_id:
        print("\nFound a car image!")
        print("Image size:", example["image"].size)
        print("Image mode:", example["image"].mode)
        break

Car label ID: 2
Car label name: Cars

Found a car image!
Image size: (275, 183)
Image mode: RGB


In [6]:
from PIL import Image
from pathlib import Path

CARS_DIR = Path(
    "/content/drive/MyDrive/PetImages/data/Neither/Cars"
)

CARS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CARS = 700
saved = 0

# Start a fresh stream
cars_dataset = load_dataset(
    "AIOmarRehan/Vehicles",
    split="train",
    streaming=True
)

label_names = cars_dataset.features["label"].names
car_label_id = label_names.index("Cars")

for example in cars_dataset:

    # Only keep images labelled Cars
    if example["label"] != car_label_id:
        continue

    image = example["image"].convert("RGB")

    save_path = CARS_DIR / f"car_{saved+1:04d}.jpg"
    image.save(save_path, "JPEG", quality=95)

    saved += 1

    if saved % 100 == 0:
        print(f"Saved {saved}/{TARGET_CARS} car images")

    if saved >= TARGET_CARS:
        break

print("\nDONE!")
print("Total car images saved:", saved)
print("Location:", CARS_DIR)

Saved 100/700 car images
Saved 200/700 car images
Saved 300/700 car images
Saved 400/700 car images
Saved 500/700 car images
Saved 600/700 car images
Saved 700/700 car images

DONE!
Total car images saved: 700
Location: /content/drive/MyDrive/PetImages/data/Neither/Cars


In [7]:
from pathlib import Path

CARS_DIR = Path(
    "/content/drive/MyDrive/PetImages/data/Neither/Cars"
)

car_images = [
    file for file in CARS_DIR.iterdir()
    if file.is_file()
]

print("Cars folder:", CARS_DIR)
print("Number of car images:", len(car_images))

print("\nFirst 10 files:")
for file in car_images[:10]:
    print("📷", file.name)

Cars folder: /content/drive/MyDrive/PetImages/data/Neither/Cars
Number of car images: 700

First 10 files:
📷 car_0001.jpg
📷 car_0002.jpg
📷 car_0003.jpg
📷 car_0004.jpg
📷 car_0005.jpg
📷 car_0006.jpg
📷 car_0007.jpg
📷 car_0008.jpg
📷 car_0009.jpg
📷 car_0010.jpg


In [8]:
from datasets import load_dataset

people_dataset = load_dataset(
    "microsoft/coco",
    split="train",
    streaming=True
)

print("Dataset loaded!")
print(people_dataset.features)

DatasetNotFoundError: Dataset 'microsoft/coco' doesn't exist on the Hub or cannot be accessed.

In [9]:
from datasets import load_dataset

people_dataset = load_dataset(
    "qianxin7274/COCO_Person",
    split="train",
    streaming=True
)

print("Dataset loaded successfully!")
print("\nFeatures:")
print(people_dataset.features)

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

Dataset loaded successfully!

Features:
{'image': Image(mode=None, decode=True), 'text': Value('string')}


In [10]:
example = next(iter(people_dataset))

image = example["image"]

print("Image size:", image.size)
print("Image mode:", image.mode)
print("Text:", example["text"])

Image size: (465, 335)
Image mode: RGB
Text: TWO PEOPLE ON A BOAT IN A RIVER.


In [11]:
from pathlib import Path
from datasets import load_dataset

PEOPLE_DIR = Path(
    "/content/drive/MyDrive/PetImages/data/Neither/People"
)

PEOPLE_DIR.mkdir(parents=True, exist_ok=True)

TARGET_PEOPLE = 700
saved = 0

# Start a fresh stream
people_dataset = load_dataset(
    "qianxin7274/COCO_Person",
    split="train",
    streaming=True
)

for example in people_dataset:

    image = example["image"].convert("RGB")

    save_path = PEOPLE_DIR / f"person_{saved+1:04d}.jpg"
    image.save(save_path, "JPEG", quality=95)

    saved += 1

    if saved % 100 == 0:
        print(f"Saved {saved}/{TARGET_PEOPLE} people images")

    if saved >= TARGET_PEOPLE:
        break

print("\nDONE!")
print("Total people images saved:", saved)
print("Location:", PEOPLE_DIR)

Saved 100/700 people images
Saved 200/700 people images
Saved 300/700 people images
Saved 400/700 people images
Saved 500/700 people images
Saved 600/700 people images
Saved 700/700 people images

DONE!
Total people images saved: 700
Location: /content/drive/MyDrive/PetImages/data/Neither/People


In [12]:
from datasets import load_dataset

food_dataset = load_dataset(
    "nateraw/food101",
    split="train",
    streaming=True
)

print("Food dataset loaded successfully!")
print("\nFeatures:")
print(food_dataset.features)

README.md:   0%|          | 0.00/3.96k [00:00<?, ?B/s]

food101.py:   0%|          | 0.00/5.20k [00:00<?, ?B/s]

RuntimeError: Dataset scripts are no longer supported, but found food101.py

In [13]:
from datasets import load_dataset

food_dataset = load_dataset(
    "ethz/food101",
    split="train",
    streaming=True
)

print("Food dataset loaded successfully!")
print("\nFeatures:")
print(food_dataset.features)

README.md:   0%|          | 0.00/16.4k [00:00<?, ?B/s]

Food dataset loaded successfully!

Features:
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare', 'beet_salad', 'beignets', 'bibimbap', 'bread_pudding', 'breakfast_burrito', 'bruschetta', 'caesar_salad', 'cannoli', 'caprese_salad', 'carrot_cake', 'ceviche', 'cheesecake', 'cheese_plate', 'chicken_curry', 'chicken_quesadilla', 'chicken_wings', 'chocolate_cake', 'chocolate_mousse', 'churros', 'clam_chowder', 'club_sandwich', 'crab_cakes', 'creme_brulee', 'croque_madame', 'cup_cakes', 'deviled_eggs', 'donuts', 'dumplings', 'edamame', 'eggs_benedict', 'escargots', 'falafel', 'filet_mignon', 'fish_and_chips', 'foie_gras', 'french_fries', 'french_onion_soup', 'french_toast', 'fried_calamari', 'fried_rice', 'frozen_yogurt', 'garlic_bread', 'gnocchi', 'greek_salad', 'grilled_cheese_sandwich', 'grilled_salmon', 'guacamole', 'gyoza', 'hamburger', 'hot_and_sour_soup', 'hot_dog', 'huevos_rancheros', 'hummus',

In [14]:
example = next(iter(food_dataset))

image = example["image"]
label_id = example["label"]
label_name = food_dataset.features["label"].names[label_id]

print("Image size:", image.size)
print("Image mode:", image.mode)
print("Food category:", label_name)

Image size: (384, 512)
Image mode: RGB
Food category: beignets


In [15]:
from pathlib import Path
from datasets import load_dataset

FOOD_DIR = Path(
    "/content/drive/MyDrive/PetImages/data/Neither/Food"
)

FOOD_DIR.mkdir(parents=True, exist_ok=True)

TARGET_FOOD = 700
saved = 0

# Start a fresh stream
food_dataset = load_dataset(
    "ethz/food101",
    split="train",
    streaming=True
)

for example in food_dataset:

    image = example["image"].convert("RGB")

    save_path = FOOD_DIR / f"food_{saved+1:04d}.jpg"
    image.save(save_path, "JPEG", quality=95)

    saved += 1

    if saved % 100 == 0:
        print(f"Saved {saved}/{TARGET_FOOD} food images")

    if saved >= TARGET_FOOD:
        break

print("\nDONE!")
print("Total food images saved:", saved)
print("Location:", FOOD_DIR)

Saved 100/700 food images
Saved 200/700 food images
Saved 300/700 food images
Saved 400/700 food images
Saved 500/700 food images
Saved 600/700 food images
Saved 700/700 food images

DONE!
Total food images saved: 700
Location: /content/drive/MyDrive/PetImages/data/Neither/Food


In [16]:
from datasets import load_dataset

buildings_dataset = load_dataset(
    "scene-15",
    split="train",
    streaming=True
)

print("Buildings dataset loaded successfully!")
print("\nFeatures:")
print(buildings_dataset.features)

DatasetNotFoundError: Dataset 'scene-15' doesn't exist on the Hub or cannot be accessed.

In [17]:
from datasets import load_dataset

buildings_dataset = load_dataset(
    "tanganke/sun397",
    split="train",
    streaming=True
)

print("Buildings dataset loaded successfully!")
print("\nFeatures:")
print(buildings_dataset.features)

README.md:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Buildings dataset loaded successfully!

Features:
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['abbey', 'airplane cabin', 'airport terminal', 'alley', 'amphitheater', 'amusement arcade', 'amusement park', 'anechoic chamber', 'apartment building outdoor', 'apse indoor', 'aquarium', 'aqueduct', 'arch', 'archive', 'arrival gate outdoor', 'art gallery', 'art school', 'art studio', 'assembly line', 'athletic field outdoor', 'atrium public', 'attic', 'auditorium', 'auto factory', 'badlands', 'badminton court indoor', 'baggage claim', 'bakery shop', 'balcony exterior', 'balcony interior', 'ball pit', 'ballroom', 'bamboo forest', 'banquet hall', 'bar', 'barn', 'barndoor', 'baseball field', 'basement', 'basilica', 'basketball court outdoor', 'bathroom', 'batters box', 'bayou', 'bazaar indoor', 'bazaar outdoor', 'beach', 'beauty salon', 'bedroom', 'berth', 'biology laboratory', 'bistro indoor', 'boardwalk', 'boat deck', 'boathouse', 'bookstore', 'booth indoor', 'botanical 

In [18]:
# Find SUN397 categories that are related to buildings

building_keywords = [
    "building",
    "house",
    "castle",
    "church",
    "cathedral",
    "mosque",
    "synagogue",
    "temple",
    "school",
    "hospital",
    "office",
    "hotel",
    "palace",
    "tower",
    "warehouse",
    "factory",
    "stadium",
    "museum",
    "library",
    "restaurant",
    "shop",
    "store",
    "apartment",
    "skyscraper",
    "courthouse",
    "monastery",
    "basilica",
    "chapel",
    "mansion",
    "cottage",
    "barn",
    "shed",
    "garage"
]

building_categories = [
    name for name in buildings_dataset.features["label"].names
    if any(keyword in name.lower() for keyword in building_keywords)
]

print("Building-related categories found:")
print()

for category in building_categories:
    print("•", category)

print()
print("Total categories:", len(building_categories))

Building-related categories found:

• apartment building outdoor
• art school
• auto factory
• bakery shop
• barn
• barndoor
• basilica
• boathouse
• bookstore
• building facade
• butchers shop
• candy store
• castle
• cathedral indoor
• cathedral outdoor
• cheese factory
• church indoor
• church outdoor
• clothing store
• coffee shop
• control tower outdoor
• cottage garden
• courthouse
• cubicle office
• dentists office
• drugstore
• factory indoor
• fastfood restaurant
• florist shop indoor
• garage indoor
• general store indoor
• general store outdoor
• gift shop
• greenhouse indoor
• greenhouse outdoor
• home office
• hospital
• hospital room
• hotel outdoor
• hotel room
• house
• jewelry shop
• library indoor
• library outdoor
• lighthouse
• mansion
• monastery outdoor
• mosque indoor
• mosque outdoor
• museum indoor
• music store
• oast house
• office
• office building
• outhouse outdoor
• palace
• parking garage indoor
• parking garage outdoor
• pilothouse indoor
• restaurant
•

In [19]:
from pathlib import Path
from PIL import Image

BUILDINGS_DIR = Path(
    "/content/drive/MyDrive/PetImages/data/Neither/Buildings"
)

BUILDINGS_DIR.mkdir(parents=True, exist_ok=True)

# Strong building/architecture categories
selected_building_categories = [
    "apartment building outdoor",
    "building facade",
    "barn",
    "basilica",
    "boathouse",
    "castle",
    "cathedral outdoor",
    "church outdoor",
    "control tower outdoor",
    "courthouse",
    "factory indoor",
    "garage indoor",
    "greenhouse outdoor",
    "hospital",
    "hotel outdoor",
    "house",
    "lighthouse",
    "mansion",
    "monastery outdoor",
    "mosque outdoor",
    "museum indoor",
    "office building",
    "palace",
    "parking garage outdoor",
    "schoolhouse",
    "shed",
    "shopfront",
    "shopping mall indoor",
    "skyscraper",
    "stadium baseball",
    "stadium football",
    "synagogue outdoor",
    "temple east asia",
    "temple south asia",
    "tower",
    "tree house",
    "warehouse indoor",
    "water tower"
]

selected_label_ids = {
    buildings_dataset.features["label"].names.index(category)
    for category in selected_building_categories
}

print("Selected categories:", len(selected_building_categories))
print("Collecting 700 building images...")

saved = 0

for item in buildings_dataset:

    if item["label"] not in selected_label_ids:
        continue

    image = item["image"]

    if image is None:
        continue

    try:
        image = image.convert("RGB")

        save_path = BUILDINGS_DIR / f"building_{saved + 1:04d}.jpg"

        image.save(
            save_path,
            "JPEG",
            quality=95
        )

        saved += 1

        if saved % 100 == 0:
            print(f"Saved {saved}/700 building images")

        if saved >= 700:
            break

    except Exception as e:
        print("Skipped image:", e)

print("\nDONE!")
print("Total building images saved:", saved)

Selected categories: 38
Saved 100/700 building images
Saved 200/700 building images
Saved 300/700 building images
Saved 400/700 building images
Saved 500/700 building images
Saved 600/700 building images
Saved 700/700 building images

DONE!
Total building images saved: 700


In [20]:
from datasets import load_dataset

landscapes_dataset = load_dataset(
    "batris-data/central-europe-rural-landscape-dataset-full",
    split="train",
    streaming=True
)

print("Landscapes dataset loaded successfully!")
print("\nFeatures:")
print(landscapes_dataset.features)

README.md:   0%|          | 0.00/9.26k [00:00<?, ?B/s]

DatasetNotFoundError: Dataset 'batris-data/central-europe-rural-landscape-dataset-full' is a gated dataset on the Hub. You must be authenticated to access it.

In [21]:
from datasets import load_dataset

landscapes_dataset = load_dataset(
    "osmr/waterfalls",
    split="train",
    streaming=True
)

print("Landscape dataset loaded successfully!")
print("\nFeatures:")
print(landscapes_dataset.features)


README.md:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/224 [00:00<?, ?it/s]

ValueError: `file_name`, `*_file_name`, `file_names` or `*_file_names` must be present as dictionary key in metadata files

In [22]:
from datasets import load_dataset

landscapes_dataset = load_dataset(
    "EmbeddingStudio/merged_remote_landscapes_v1",
    split="train",
    streaming=True
)

print("Landscapes dataset loaded successfully!")
print("\nFeatures:")
print(landscapes_dataset.features)

README.md:   0%|          | 0.00/3.56k [00:00<?, ?B/s]

Landscapes dataset loaded successfully!

Features:
{'image': Image(mode=None, decode=True), 'category': Value('string'), 'img_id': Value('string')}


In [23]:
# Collect the unique landscape categories without loading images into RAM

landscape_categories = set()

for item in landscapes_dataset:
    landscape_categories.add(item["category"])

print("Landscape categories found:")
print()

for category in sorted(landscape_categories):
    print("•", category)

print()
print("Total categories:", len(landscape_categories))

Landscape categories found:

• agricultural
• airplane
• airport
• baseball diamond
• basketball court
• beach
• bridge
• buildings
• chaparral
• church
• circular farmland
• cloud
• commercial area
• desert
• forest
• freeway
• golf course
• ground track field
• harbor
• industrial area
• intersection
• island
• lake
• meadow
• mountain
• overpass
• palace
• parking lot
• railway
• railway station
• rectangular farmland
• residential
• river
• roundabout
• runway
• sea ice
• ship
• snowberg
• stadium
• storage tanks
• tennis court
• terrace
• thermal power station
• wetland

Total categories: 44


In [24]:
from pathlib import Path

LANDSCAPES_DIR = Path(
    "/content/drive/MyDrive/PetImages/data/Neither/Landscapes"
)

LANDSCAPES_DIR.mkdir(parents=True, exist_ok=True)

# Natural landscape categories
selected_landscape_categories = [
    "agricultural",
    "beach",
    "chaparral",
    "circular farmland",
    "cloud",
    "desert",
    "forest",
    "island",
    "lake",
    "meadow",
    "mountain",
    "river",
    "sea ice",
    "snowberg",
    "wetland"
]

print("Selected categories:", len(selected_landscape_categories))
print("Collecting 700 landscape images...")

saved = 0

for item in landscapes_dataset:

    if item["category"] not in selected_landscape_categories:
        continue

    image = item["image"]

    if image is None:
        continue

    try:
        image = image.convert("RGB")

        save_path = LANDSCAPES_DIR / f"landscape_{saved + 1:04d}.jpg"

        image.save(
            save_path,
            "JPEG",
            quality=95
        )

        saved += 1

        if saved % 100 == 0:
            print(f"Saved {saved}/700 landscape images")

        if saved >= 700:
            break

    except Exception as e:
        print("Skipped image:", e)

print("\nDONE!")
print("Total landscape images saved:", saved)

Selected categories: 15
Saved 100/700 landscape images
Saved 200/700 landscape images
Saved 300/700 landscape images
Saved 400/700 landscape images
Saved 500/700 landscape images
Saved 600/700 landscape images
Saved 700/700 landscape images

DONE!
Total landscape images saved: 700


In [25]:
from datasets import load_dataset

animals_dataset = load_dataset(
    "LibreYOLO/animals-ij5d2",
    split="train",
    streaming=True
)

print("Animal dataset loaded successfully!")
print("\nFeatures:")
print(animals_dataset.features)

README.md:   0%|          | 0.00/2.96k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/1400 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/400 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/200 [00:00<?, ?it/s]

Animal dataset loaded successfully!

Features:
{'image': Image(mode=None, decode=True), 'label': Value('string')}


In [26]:
animal_categories = set()

for item in animals_dataset:
    animal_categories.add(item["label"])

print("Animal categories found:")
print()

for category in sorted(animal_categories):
    print("•", category)

print()
print("Total categories:", len(animal_categories))

CastError: Couldn't cast
text: string
image: null
label: null
to
{'image': Image(mode=None, decode=True), 'label': Value('string')}
because column names don't match

In [27]:
from datasets import load_dataset

animals_dataset = load_dataset(
    "galilai-group/awa2",
    split="test",
    streaming=True
)

print("Animal dataset loaded successfully!")
print("\nFeatures:")
print(animals_dataset.features)

README.md:   0%|          | 0.00/3.21k [00:00<?, ?B/s]

awa2.py:   0%|          | 0.00/4.32k [00:00<?, ?B/s]

RuntimeError: Dataset scripts are no longer supported, but found awa2.py

In [28]:
from datasets import load_dataset

animals_dataset = load_dataset(
    "chriamue/animal10n",
    split="train",
    streaming=True
)

print("Animal dataset loaded successfully!")
print("\nFeatures:")
print(animals_dataset.features)

DatasetNotFoundError: Dataset 'chriamue/animal10n' doesn't exist on the Hub or cannot be accessed.

In [29]:
from datasets import load_dataset

animals_dataset = load_dataset(
    "lucabaggi/animal-wildlife",
    split="train",
    streaming=True
)

print("Animal dataset loaded successfully!")
print("\nFeatures:")
print(animals_dataset.features)

README.md:   0%|          | 0.00/5.03k [00:00<?, ?B/s]

Animal dataset loaded successfully!

Features:
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['antelope', 'badger', 'bat', 'bear', 'bee', 'beetle', 'bison', 'boar', 'butterfly', 'cat', 'caterpillar', 'chimpanzee', 'cockroach', 'cow', 'coyote', 'crab', 'crow', 'deer', 'dog', 'dolphin', 'donkey', 'dragonfly', 'duck', 'eagle', 'elephant', 'flamingo', 'fly', 'fox', 'goat', 'goldfish', 'goose', 'gorilla', 'grasshopper', 'hamster', 'hare', 'hedgehog', 'hippopotamus', 'hornbill', 'horse', 'hummingbird', 'hyena', 'jellyfish', 'kangaroo', 'koala', 'ladybugs', 'leopard', 'lion', 'lizard', 'lobster', 'mosquito', 'moth', 'mouse', 'octopus', 'okapi', 'orangutan', 'otter', 'owl', 'ox', 'oyster', 'panda', 'parrot', 'pelecaniformes', 'penguin', 'pig', 'pigeon', 'porcupine', 'possum', 'raccoon', 'rat', 'reindeer', 'rhinoceros', 'sandpiper', 'seahorse', 'seal', 'shark', 'sheep', 'snake', 'sparrow', 'squid', 'squirrel', 'starfish', 'swan', 'tiger', 'turkey', 'turtle', 'whale', 'wolf'

In [30]:
from pathlib import Path

OTHER_ANIMALS_DIR = Path(
    "/content/drive/MyDrive/PetImages/data/Neither/Other_Animals"
)

OTHER_ANIMALS_DIR.mkdir(parents=True, exist_ok=True)

# All animal categories except cats and dogs
excluded_animals = {"cat", "dog"}

animal_label_names = animals_dataset.features["label"].names

allowed_animal_labels = {
    label_id
    for label_id, label_name in enumerate(animal_label_names)
    if label_name not in excluded_animals
}

print("Excluded:", excluded_animals)
print("Allowed animal categories:", len(allowed_animal_labels))
print("Collecting 700 other-animal images...")

saved = 0

for item in animals_dataset:

    if item["label"] not in allowed_animal_labels:
        continue

    image = item["image"]

    if image is None:
        continue

    try:
        image = image.convert("RGB")

        save_path = (
            OTHER_ANIMALS_DIR /
            f"animal_{saved + 1:04d}.jpg"
        )

        image.save(
            save_path,
            "JPEG",
            quality=95
        )

        saved += 1

        if saved % 100 == 0:
            print(f"Saved {saved}/700 other-animal images")

        if saved >= 700:
            break

    except Exception as e:
        print("Skipped image:", e)

print("\nDONE!")
print("Total other-animal images saved:", saved)

Excluded: {'dog', 'cat'}
Allowed animal categories: 88
Saved 100/700 other-animal images
Saved 200/700 other-animal images
Saved 300/700 other-animal images
Saved 400/700 other-animal images
Saved 500/700 other-animal images
Saved 600/700 other-animal images
Saved 700/700 other-animal images

DONE!
Total other-animal images saved: 700


In [31]:
from datasets import load_dataset

objects_dataset = load_dataset(
    "cifar10",
    split="train",
    streaming=True
)

print("Objects dataset loaded successfully!")
print("\nFeatures:")
print(objects_dataset.features)


README.md:   0%|          | 0.00/5.16k [00:00<?, ?B/s]

HfUriError: Invalid HF URI 'hf://datasets/cifar10@0b2714987fa478483af9968de7c934580d0bb9a2/.huggingface.yaml'. Repository id must be 'namespace/name', got 'cifar10'.

In [32]:
from datasets import load_dataset

objects_dataset = load_dataset(
    "zh-plus/object-detection",
    split="train",
    streaming=True
)

print("Objects dataset loaded successfully!")
print("\nFeatures:")
print(objects_dataset.features)

DatasetNotFoundError: Dataset 'zh-plus/object-detection' doesn't exist on the Hub or cannot be accessed.

In [33]:
from datasets import load_dataset

objects_dataset = load_dataset(
    "unstructured/pix3d",
    split="train",
    streaming=True
)

print("Objects dataset loaded successfully!")
print("\nFeatures:")
print(objects_dataset.features)

README.md:   0%|          | 0.00/6.56k [00:00<?, ?B/s]

DataFilesNotFoundError: No (supported) data files found in unstructured/pix3d

In [34]:
from datasets import load_dataset

objects_dataset = load_dataset(
    "thethinkmachine/tiny-imagenet",
    split="train",
    streaming=True
)

print("Objects dataset loaded successfully!")
print("\nFeatures:")
print(objects_dataset.features)

README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

Objects dataset loaded successfully!

Features:
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['n01443537', 'n01629819', 'n01641577', 'n01644900', 'n01698640', 'n01742172', 'n01768244', 'n01770393', 'n01774384', 'n01774750', 'n01784675', 'n01882714', 'n01910747', 'n01917289', 'n01944390', 'n01950731', 'n01983481', 'n01984695', 'n02002724', 'n02056570', 'n02058221', 'n02074367', 'n02094433', 'n02099601', 'n02099712', 'n02106662', 'n02113799', 'n02123045', 'n02123394', 'n02124075', 'n02125311', 'n02129165', 'n02132136', 'n02165456', 'n02226429', 'n02231487', 'n02233338', 'n02236044', 'n02268443', 'n02279972', 'n02281406', 'n02321529', 'n02364673', 'n02395406', 'n02403003', 'n02410509', 'n02415577', 'n02423022', 'n02437312', 'n02480495', 'n02481823', 'n02486410', 'n02504458', 'n02509815', 'n02666347', 'n02669723', 'n02699494', 'n02769748', 'n02788148', 'n02791270', 'n02793495', 'n02795169', 'n02802426', 'n02808440', 'n02814533', 'n02814860', 'n02815834', 'n02823428', 

In [35]:
# Decode Tiny ImageNet WordNet IDs into readable class names

from datasets import load_dataset

label_names = objects_dataset.features["label"].names

# Tiny ImageNet class names
tiny_imagenet_labels = {
    "n01443537": "tench",
    "n01629819": "European fire salamander",
    "n01641577": "bullfrog",
    "n01644900": "tailed frog",
    "n01698640": "American alligator",
    "n01742172": "boa constrictor",
    "n01768244": "trilobite",
    "n01770393": "scorpion",
    "n01774384": "black widow",
    "n01774750": "tarantula",
    "n01784675": "centipede",
    "n01882714": "koala",
    "n01910747": "jellyfish",
    "n01917289": "brain coral",
    "n01944390": "snail",
    "n01950731": "slug",
    "n01983481": "American lobster",
    "n01984695": "spiny lobster",
    "n02002724": "black stork",
    "n02056570": "king penguin",
    "n02058221": "albatross",
    "n02074367": "dugong",
    "n02094433": "brambling",
    "n02099601": "goldfinch",
    "n02099712": "junco",
    "n02106662": "water ouzel",
    "n02113799": "black-backed gull",
    "n02123045": "tabby cat",
    "n02123394": "Persian cat",
    "n02124075": "Egyptian cat",
    "n02125311": "cougar",
    "n02129165": "lion",
    "n02132136": "brown bear",
    "n02165456": "ladybug",
    "n02226429": "grasshopper",
    "n02231487": "walking stick",
    "n02233338": "cockroach",
    "n02236044": "mantis",
    "n02268443": "dragonfly",
    "n02279972": "monarch butterfly",
    "n02281406": "sulphur butterfly",
    "n02321529": "sea cucumber",
    "n02364673": "guinea pig",
    "n02395406": "hog",
    "n02403003": "ox",
    "n02410509": "bison",
    "n02415577": "bighorn sheep",
    "n02423022": "gazelle",
    "n02437312": "Arabian camel",
    "n02480495": "orangutan",
    "n02481823": "chimpanzee",
    "n02486410": "baboon",
    "n02504458": "African elephant",
    "n02509815": "lesser panda",
    "n02666347": "abaya",
    "n02669723": "academic gown",
    "n02699494": "altar",
    "n02769748": "backpack",
    "n02788148": "ball",
    "n02791270": "barrow",
    "n02793495": "bell cote",
    "n02795169": "bin",
    "n02802426": "basket",
    "n02808440": "bathtub",
    "n02814533": "beach wagon",
    "n02814860": "beacon",
    "n02815834": "beaker",
    "n02823428": "beer bottle",
    "n02837789": "bikini",
    "n02841315": "binoculars",
    "n02843684": "birdhouse",
    "n02883205": "bow tie",
    "n02892201": "brassiere",
    "n02909870": "bucket",
    "n02917067": "bulletproof vest",
    "n02927161": "butcher shop",
    "n02948072": "candle",
    "n02950826": "cannon",
    "n02963159": "cardigan",
    "n02977067": "cash machine",
    "n02988304": "CD player",
    "n03014705": "chest",
    "n03026506": "Christmas stocking",
    "n03042490": "cliff dwelling",
    "n03085013": "computer",
    "n03089624": "confectionery",
    "n03100240": "convertible",
    "n03126707": "crane",
    "n03160309": "dam",
    "n03179701": "desk",
    "n03201208": "dishtowel",
    "n03255030": "dumbbell",
    "n03355925": "flag",
    "n03373237": "folding chair",
    "n03388043": "folding screen",
    "n03393912": "forklift",
    "n03400231": "fur coat",
    "n03404251": "garbage truck",
    "n03424325": "gasmask",
    "n03444034": "go-kart",
    "n03447447": "gondola",
    "n03544143": "hourglass",
    "n03584254": "iPod",
    "n03599486": "jinrikisha",
    "n03617480": "kimono",
    "n03637318": "lampshade",
    "n03649909": "lawn mower",
    "n03662601": "lifeboat",
    "n03670208": "limousine",
    "n03706229": "magnetic compass",
    "n03733131": "maypole",
    "n03763968": "military uniform",
    "n03770439": "miniskirt",
    "n03796401": "moving van",
    "n03814639": "neck brace",
    "n03837869": "obelisk",
    "n03838899": "oboe",
    "n03854065": "organ",
    "n03891332": "parking meter",
    "n03902125": "pay-phone",
    "n03930313": "picket fence",
    "n03937543": "pill bottle",
    "n03970156": "plunger",
    "n03977966": "police van",
    "n03980874": "poncho",
    "n03983396": "pop bottle",
    "n03992509": "pot",
    "n04008634": "projectile",
    "n04023962": "punching bag",
    "n04070727": "refrigerator",
    "n04074963": "remote control",
    "n04099969": "rocking chair",
    "n04118538": "rug",
    "n04133789": "sandal",
    "n04146614": "school bus",
    "n04149813": "scoreboard",
    "n04179913": "sewing machine",
    "n04251144": "snorkel",
    "n04254777": "sock",
    "n04259630": "sombrero",
    "n04265275": "space heater",
    "n04275548": "spatula",
    "n04285008": "sports car",
    "n04311004": "steel arch bridge",
    "n04328186": "stopwatch",
    "n04356056": "sunglasses",
    "n04366367": "suspension bridge",
    "n04371430": "swimming trunks",
    "n04376876": "syringe",
    "n04398044": "table lamp",
    "n04399382": "teddy bear",
    "n04417672": "thatch",
    "n04456115": "torch",
    "n04465666": "tractor",
    "n04486054": "triumphal arch",
    "n04487081": "trolleybus",
    "n04501370": "turnstile",
    "n04507155": "umbrella",
    "n04532106": "vestment",
    "n04532670": "viaduct",
    "n04540053": "volleyball",
    "n04560804": "water bottle",
    "n04562935": "watercraft",
    "n04596742": "wok",
    "n04598010": "wooden spoon",
    "n06596364": "comic book",
    "n07056680": "crossword puzzle",
    "n07583066": "guacamole",
    "n07614500": "ice cream",
    "n07615774": "ice lolly",
    "n07646821": "pizza",
    "n07647870": "plate",
    "n07657664": "rain barrel",
    "n07695742": "cardoon",
    "n07711569": "mashed potato",
    "n07715103": "cauliflower",
    "n07720875": "bell pepper",
    "n07749582": "lemon",
    "n07753592": "banana",
    "n07768694": "pomegranate",
    "n07871810": "pretzel",
    "n07873807": "pizza",
    "n07875152": "potpie",
    "n07920052": "espresso",
    "n07975909": "root beer",
    "n08496334": "clothing",
    "n08620881": "stage",
    "n08742578": "apiary",
    "n09193705": "alp",
    "n09246464": "cliff",
    "n09256479": "coral reef",
    "n09332890": "lakeside",
    "n09428293": "seashore",
    "n12267677": "acorn",
    "n12520864": "hip",
    "n13001041": "bolete",
    "n13652335": "rapeseed",
    "n13652994": "corn",
    "n13719102": "yard",
    "n14991210": "toilet tissue"
}

print("Decoded labels:\n")

for label_id in label_names:
    name = tiny_imagenet_labels.get(label_id, "UNKNOWN")
    print(f"{label_id} → {name}")

Decoded labels:

n01443537 → tench
n01629819 → European fire salamander
n01641577 → bullfrog
n01644900 → tailed frog
n01698640 → American alligator
n01742172 → boa constrictor
n01768244 → trilobite
n01770393 → scorpion
n01774384 → black widow
n01774750 → tarantula
n01784675 → centipede
n01882714 → koala
n01910747 → jellyfish
n01917289 → brain coral
n01944390 → snail
n01950731 → slug
n01983481 → American lobster
n01984695 → spiny lobster
n02002724 → black stork
n02056570 → king penguin
n02058221 → albatross
n02074367 → dugong
n02094433 → brambling
n02099601 → goldfinch
n02099712 → junco
n02106662 → water ouzel
n02113799 → black-backed gull
n02123045 → tabby cat
n02123394 → Persian cat
n02124075 → Egyptian cat
n02125311 → cougar
n02129165 → lion
n02132136 → brown bear
n02165456 → ladybug
n02226429 → grasshopper
n02231487 → walking stick
n02233338 → cockroach
n02236044 → mantis
n02268443 → dragonfly
n02279972 → monarch butterfly
n02281406 → sulphur butterfly
n02321529 → sea cucumber
n0236

In [36]:
from pathlib import Path

OBJECTS_DIR = Path(
    "/content/drive/MyDrive/PetImages/data/Neither/Objects"
)

OBJECTS_DIR.mkdir(parents=True, exist_ok=True)

# 28 object categories × 25 images = exactly 700 images
selected_object_categories = [
    "backpack",
    "bin",
    "basket",
    "bathtub",
    "beaker",
    "binoculars",
    "candle",
    "CD player",
    "computer",
    "desk",
    "dumbbell",
    "folding chair",
    "lampshade",
    "magnetic compass",
    "oboe",
    "parking meter",
    "pay-phone",
    "pill bottle",
    "plunger",
    "refrigerator",
    "remote control",
    "rocking chair",
    "sewing machine",
    "stopwatch",
    "umbrella",
    "wok",
    "wooden spoon",
    "toilet tissue"
]

IMAGES_PER_CATEGORY = 25

# Convert readable names → Tiny ImageNet label IDs
name_to_id = {
    name: objects_dataset.features["label"].names.index(wnid)
    for wnid, name in tiny_imagenet_labels.items()
    if name in selected_object_categories
}

print("Selected object categories:", len(name_to_id))
print("Images per category:", IMAGES_PER_CATEGORY)
print("Target:", len(name_to_id) * IMAGES_PER_CATEGORY)

# Track how many images we've collected from each category
category_counts = {
    label_id: 0
    for label_id in name_to_id.values()
}

id_to_name = {
    label_id: name
    for name, label_id in name_to_id.items()
}

saved = 0

for item in objects_dataset:

    label_id = item["label"]

    # Ignore categories we don't want
    if label_id not in category_counts:
        continue

    # Each category gets exactly 25 images
    if category_counts[label_id] >= IMAGES_PER_CATEGORY:
        continue

    image = item["image"]

    if image is None:
        continue

    try:
        image = image.convert("RGB")

        save_path = (
            OBJECTS_DIR /
            f"object_{saved + 1:04d}.jpg"
        )

        image.save(
            save_path,
            "JPEG",
            quality=95
        )

        category_counts[label_id] += 1
        saved += 1

        if saved % 100 == 0:
            print(f"Saved {saved}/700 object images")

        # Stop once every selected category has 25
        if saved >= 700:
            break

    except Exception as e:
        print("Skipped image:", e)

print("\nDONE!")
print("Total object images saved:", saved)

print("\nImages collected per category:")
for label_id, count in category_counts.items():
    print(f"• {id_to_name[label_id]}: {count}")

Selected object categories: 28
Images per category: 25
Target: 700
Saved 100/700 object images
Saved 200/700 object images
Saved 300/700 object images
Saved 400/700 object images
Saved 500/700 object images
Saved 600/700 object images
Saved 700/700 object images

DONE!
Total object images saved: 700

Images collected per category:
• backpack: 25
• bin: 25
• basket: 25
• bathtub: 25
• beaker: 25
• binoculars: 25
• candle: 25
• CD player: 25
• computer: 25
• desk: 25
• dumbbell: 25
• folding chair: 25
• lampshade: 25
• magnetic compass: 25
• oboe: 25
• parking meter: 25
• pay-phone: 25
• pill bottle: 25
• plunger: 25
• refrigerator: 25
• remote control: 25
• rocking chair: 25
• sewing machine: 25
• stopwatch: 25
• umbrella: 25
• wok: 25
• wooden spoon: 25
• toilet tissue: 25


In [37]:
from pathlib import Path
import random

# Dataset folders
DATA_DIR = Path("/content/drive/MyDrive/PetImages/data")

CAT_DIR = DATA_DIR / "Cat"
DOG_DIR = DATA_DIR / "Dog"
NEITHER_DIR = DATA_DIR / "Neither"

# Collect image paths
cat_images = list(CAT_DIR.glob("*.jpg"))
dog_images = list(DOG_DIR.glob("*.jpg"))

neither_images = []
for folder in NEITHER_DIR.iterdir():
    if folder.is_dir():
        neither_images.extend(folder.glob("*.jpg"))

print("Available images:")
print("CAT:", len(cat_images))
print("DOG:", len(dog_images))
print("NEITHER:", len(neither_images))

# We want equal numbers from all 3 classes
TARGET_PER_CLASS = 4900

# Make the selection reproducible
random.seed(42)

cat_selected = random.sample(cat_images, TARGET_PER_CLASS)
dog_selected = random.sample(dog_images, TARGET_PER_CLASS)
neither_selected = random.sample(neither_images, TARGET_PER_CLASS)

# Create paths and labels
image_paths = (
    cat_selected +
    dog_selected +
    neither_selected
)

labels = (
    [0] * TARGET_PER_CLASS +   # CAT
    [1] * TARGET_PER_CLASS +   # DOG
    [2] * TARGET_PER_CLASS     # NEITHER
)

# Shuffle everything together
combined = list(zip(image_paths, labels))
random.shuffle(combined)

image_paths, labels = zip(*combined)

image_paths = list(image_paths)
labels = list(labels)

print("\nBalanced dataset created!")
print("Total images:", len(image_paths))

print("\nClass distribution:")
print("CAT:", labels.count(0))
print("DOG:", labels.count(1))
print("NEITHER:", labels.count(2))

Available images:
CAT: 12499
DOG: 12499
NEITHER: 4900

Balanced dataset created!
Total images: 14700

Class distribution:
CAT: 4900
DOG: 4900
NEITHER: 4900


In [38]:
from sklearn.model_selection import train_test_split

# Split while preserving the 3-class balance
train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

print("Training images:", len(train_paths))
print("Validation images:", len(val_paths))

print("\nTraining distribution:")
print("CAT:", train_labels.count(0))
print("DOG:", train_labels.count(1))
print("NEITHER:", train_labels.count(2))

print("\nValidation distribution:")
print("CAT:", val_labels.count(0))
print("DOG:", val_labels.count(1))
print("NEITHER:", val_labels.count(2))

Training images: 11760
Validation images: 2940

Training distribution:
CAT: 3920
DOG: 3920
NEITHER: 3920

Validation distribution:
CAT: 980
DOG: 980
NEITHER: 980


In [39]:
from torch.utils.data import Dataset
from PIL import Image
import torch


class CatDogNeitherDataset(Dataset):

    def __init__(self, image_paths, labels, processor):
        self.image_paths = image_paths
        self.labels = labels
        self.processor = processor

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image = Image.open(
            self.image_paths[idx]
        ).convert("RGB")

        processed = self.processor(
            images=image,
            return_tensors="pt"
        )

        pixel_values = processed["pixel_values"].squeeze(0)

        label = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return {
            "pixel_values": pixel_values,
            "labels": label
        }


print("Dataset class created successfully!")

Dataset class created successfully!


In [40]:
train_dataset = CatDogNeitherDataset(
    train_paths,
    train_labels,
    processor
)

val_dataset = CatDogNeitherDataset(
    val_paths,
    val_labels,
    processor
)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

NameError: name 'processor' is not defined

In [41]:
from transformers import AutoImageProcessor

MODEL_PATH = "/content/drive/MyDrive/PetImages/cat_dog_resnet50"

processor = AutoImageProcessor.from_pretrained(MODEL_PATH)

print("Processor loaded successfully!")

Processor loaded successfully!


In [42]:
train_dataset = CatDogNeitherDataset(
    train_paths,
    train_labels,
    processor
)

val_dataset = CatDogNeitherDataset(
    val_paths,
    val_labels,
    processor
)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

Training dataset: 11760
Validation dataset: 2940


In [43]:
from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders created successfully!")
print("Batch size:", BATCH_SIZE)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

DataLoaders created successfully!
Batch size: 16
Training batches: 735
Validation batches: 184


In [44]:
# Get one batch
batch = next(iter(train_loader))

print("Pixel values shape:", batch["pixel_values"].shape)
print("Labels shape:", batch["labels"].shape)
print("Labels in this batch:", batch["labels"].tolist())

# Move the batch to GPU
batch = {
    key: value.to(device)
    for key, value in batch.items()
}

print("\nGPU test successful!")
print("Pixel values device:", batch["pixel_values"].device)
print("Labels device:", batch["labels"].device)

Pixel values shape: torch.Size([16, 3, 224, 224])
Labels shape: torch.Size([16])
Labels in this batch: [2, 1, 0, 1, 1, 2, 1, 0, 2, 1, 1, 2, 0, 2, 2, 2]


NameError: name 'device' is not defined

In [45]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [46]:
# Move the test batch to the GPU
batch = {
    key: value.to(device)
    for key, value in batch.items()
}

print("GPU test successful!")
print("Pixel values device:", batch["pixel_values"].device)
print("Labels device:", batch["labels"].device)

GPU test successful!
Pixel values device: cuda:0
Labels device: cuda:0


In [47]:
import torch
from transformers import ResNetForImageClassification

MODEL_PATH = "/content/drive/MyDrive/PetImages/cat_dog_resnet50"

# Load your previously trained Cat/Dog model
model = ResNetForImageClassification.from_pretrained(
    MODEL_PATH
)

# Replace the old 2-class classifier with a new 3-class classifier
model.classifier = torch.nn.Linear(
    model.classifier.in_features,
    3
)

# Update the model's label information
model.config.num_labels = 3

model.config.id2label = {
    0: "CAT",
    1: "DOG",
    2: "NEITHER"
}

model.config.label2id = {
    "CAT": 0,
    "DOG": 1,
    "NEITHER": 2
}

# Move model to GPU
model = model.to(device)

print("3-class model ready!")
print("Number of classes:", model.config.num_labels)
print("Labels:", model.config.id2label)
print("Device:", next(model.parameters()).device)

Loading weights:   0%|          | 0/320 [00:01<?, ?it/s]

AttributeError: 'Sequential' object has no attribute 'in_features'

In [48]:
print(model.classifier)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=2048, out_features=2, bias=True)
)


In [49]:
# Replace the 2-class Linear layer with a 3-class Linear layer
model.classifier[1] = torch.nn.Linear(
    in_features=2048,
    out_features=3
)

# Update label information
model.config.num_labels = 3

model.config.id2label = {
    0: "CAT",
    1: "DOG",
    2: "NEITHER"
}

model.config.label2id = {
    "CAT": 0,
    "DOG": 1,
    "NEITHER": 2
}

# Make sure the model is on the GPU
model = model.to(device)

print("3-class model ready!")
print("\nClassifier:")
print(model.classifier)

print("\nLabels:")
print(model.config.id2label)

print("\nDevice:", next(model.parameters()).device)

3-class model ready!

Classifier:
Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=2048, out_features=3, bias=True)
)

Labels:
{0: 'CAT', 1: 'DOG', 2: 'NEITHER'}

Device: cuda:0


In [50]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=5e-5
)

print("Optimizer ready!")
print("Learning rate:", 5e-5)

Optimizer ready!
Learning rate: 5e-05


In [51]:
model.train()

total_loss = 0

for batch_idx, batch in enumerate(train_loader):

    # Move batch to GPU
    batch = {
        key: value.to(device)
        for key, value in batch.items()
    }

    # Clear previous gradients
    optimizer.zero_grad()

    # Forward pass
    outputs = model(**batch)

    # Calculate loss
    loss = outputs.loss

    # Backpropagation
    loss.backward()

    # Update model weights
    optimizer.step()

    total_loss += loss.item()

    # Show progress every 100 batches
    if (batch_idx + 1) % 100 == 0:
        print(
            f"Batch {batch_idx + 1}/{len(train_loader)} "
            f"- Loss: {loss.item():.4f}"
        )

average_loss = total_loss / len(train_loader)

print("\nEpoch 1 complete!")
print("Average training loss:", average_loss)

Batch 100/735 - Loss: 0.5900
Batch 200/735 - Loss: 0.3257
Batch 300/735 - Loss: 0.4619
Batch 400/735 - Loss: 0.1142
Batch 500/735 - Loss: 0.0872
Batch 600/735 - Loss: 0.0294
Batch 700/735 - Loss: 0.1168

Epoch 1 complete!
Average training loss: 0.2886847152824507


In [52]:
# Save the trained 3-class model to Google Drive

MODEL_SAVE_PATH = (
    "/content/drive/MyDrive/PetImages/"
    "cat_dog_neither_resnet50"
)

model.save_pretrained(MODEL_SAVE_PATH)
processor.save_pretrained(MODEL_SAVE_PATH)

print("Model saved successfully!")
print("Saved to:", MODEL_SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!
Saved to: /content/drive/MyDrive/PetImages/cat_dog_neither_resnet50


In [54]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch_idx, batch in enumerate(val_loader):

        # Move batch to GPU
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        # Make predictions
        outputs = model(
            pixel_values=batch["pixel_values"]
        )

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        # Count correct predictions
        correct += (
            predictions == batch["labels"]
        ).sum().item()

        total += batch["labels"].size(0)

        if (batch_idx + 1) % 25 == 0:
            print(
                f"Validation batch "
                f"{batch_idx + 1}/{len(val_loader)}"
            )

accuracy = correct / total

print("\nValidation complete!")
print("Correct:", correct)
print("Total:", total)
print(f"Validation accuracy: {accuracy * 100:.2f}%")

Validation batch 25/184
Validation batch 50/184
Validation batch 75/184
Validation batch 100/184
Validation batch 125/184
Validation batch 150/184
Validation batch 175/184

Validation complete!
Correct: 2907
Total: 2940
Validation accuracy: 98.88%


In [55]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in val_loader:

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = model(
            pixel_values=batch["pixel_values"]
        )

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            batch["labels"].cpu().numpy()
        )


# Classification report
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "CAT",
            "DOG",
            "NEITHER"
        ],
        digits=4
    )
)


# Confusion matrix
cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("\nCONFUSION MATRIX")
print("=" * 60)

print(cm)

print("\nRows = Actual")
print("Columns = Predicted")
print("\n       CAT  DOG  NEITHER")
print("CAT   ", cm[0])
print("DOG   ", cm[1])
print("NEITHER", cm[2])

CLASSIFICATION REPORT
              precision    recall  f1-score   support

         CAT     0.9959    0.9827    0.9892       980
         DOG     0.9888    0.9898    0.9893       980
     NEITHER     0.9819    0.9939    0.9878       980

    accuracy                         0.9888      2940
   macro avg     0.9888    0.9888    0.9888      2940
weighted avg     0.9888    0.9888    0.9888      2940


CONFUSION MATRIX
[[963   7  10]
 [  2 970   8]
 [  2   4 974]]

Rows = Actual
Columns = Predicted

       CAT  DOG  NEITHER
CAT    [963   7  10]
DOG    [  2 970   8]
NEITHER [  2   4 974]


In [56]:
def predict_3class(image):

    # Prepare the image
    inputs = processor(
        images=image,
        return_tensors="pt"
    )

    # Move inputs to GPU
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Evaluation mode
    model.eval()

    # Make prediction
    with torch.no_grad():

        outputs = model(**inputs)

        probabilities = torch.softmax(
            outputs.logits,
            dim=1
        )

    # Get predicted class
    predicted_class = torch.argmax(
        probabilities,
        dim=1
    ).item()

    # Get confidence
    confidence = probabilities[0][predicted_class].item()

    # Get all probabilities
    cat_probability = probabilities[0][0].item()
    dog_probability = probabilities[0][1].item()
    neither_probability = probabilities[0][2].item()

    # Convert class number to label
    label = model.config.id2label[predicted_class]

    return {
        "prediction": label,
        "confidence": confidence,
        "CAT": cat_probability,
        "DOG": dog_probability,
        "NEITHER": neither_probability
    }


print("3-class prediction function ready!")

3-class prediction function ready!


In [57]:
from PIL import Image

test_image_path = (
    "/content/drive/MyDrive/PetImages/data/Dog/6511.jpg"
)

test_image = Image.open(test_image_path).convert("RGB")

result = predict_3class(test_image)

print("Prediction:", result["prediction"])
print(f"Confidence: {result['confidence'] * 100:.2f}%")

print("\nClass probabilities:")
print(f"CAT:     {result['CAT'] * 100:.2f}%")
print(f"DOG:     {result['DOG'] * 100:.2f}%")
print(f"NEITHER: {result['NEITHER'] * 100:.2f}%")

Prediction: DOG
Confidence: 99.36%

Class probabilities:
CAT:     0.26%
DOG:     99.36%
NEITHER: 0.38%


In [58]:
from pathlib import Path
from PIL import Image

car_dir = Path(
    "/content/drive/MyDrive/PetImages/data/Neither/Cars"
)

car_images = list(car_dir.glob("*.jpg"))

print("Car images available:", len(car_images))

car_image = Image.open(
    car_images[0]
).convert("RGB")

car_result = predict_3class(car_image)

print("\nPrediction:", car_result["prediction"])
print(f"Confidence: {car_result['confidence'] * 100:.2f}%")

print("\nClass probabilities:")
print(f"CAT:     {car_result['CAT'] * 100:.2f}%")
print(f"DOG:     {car_result['DOG'] * 100:.2f}%")
print(f"NEITHER: {car_result['NEITHER'] * 100:.2f}%")

Car images available: 700

Prediction: NEITHER
Confidence: 98.78%

Class probabilities:
CAT:     0.41%
DOG:     0.81%
NEITHER: 98.78%


In [59]:
import gradio as gr


def classify_3class(image):

    result = predict_3class(image)

    return {
        "CAT": result["CAT"],
        "DOG": result["DOG"],
        "NEITHER": result["NEITHER"]
    }


app = gr.Interface(
    fn=classify_3class,

    inputs=gr.Image(
        type="pil",
        label="Upload an image"
    ),

    outputs=gr.Label(
        num_top_classes=3,
        label="AI Prediction"
    ),

    title="🐱 Cat vs Dog vs Neither 🐶",

    description=(
        "Upload an image and the AI will predict whether "
        "it contains a CAT, DOG, or NEITHER."
    )
)

print("Gradio app created successfully!")

Gradio app created successfully!


In [60]:
app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://36e82e63b132665368.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
